In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# =========================
# LOAD DATA
# =========================

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

# Save PassengerId
test_passenger_ids = test["PassengerId"]

# =========================
# FEATURE ENGINEERING
# =========================

# Family Size
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

# Is Alone
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

# Extract Title from Name
train["Title"] = train["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

# Group rare titles
rare_titles = [
    "Lady", "Countess", "Capt", "Col", "Don",
    "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"
]

train["Title"] = train["Title"].replace(rare_titles, "Rare")
test["Title"] = test["Title"].replace(rare_titles, "Rare")

train["Title"] = train["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

test["Title"] = test["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

# =========================
# HANDLE MISSING VALUES
# =========================

train["Age"] = train["Age"].fillna(train["Age"].median())
test["Age"] = test["Age"].fillna(test["Age"].median())

train["Embarked"] = train["Embarked"].fillna(
    train["Embarked"].mode()[0]
)

test["Fare"] = test["Fare"].fillna(
    test["Fare"].median()
)

# =========================
# DROP UNUSED COLUMNS
# =========================

drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

# =========================
# ENCODE CATEGORICAL DATA
# =========================

combined = pd.concat(
    [train.drop("Survived", axis=1), test],
    axis=0
)

combined = pd.get_dummies(
    combined,
    columns=["Sex", "Embarked", "Title"],
    drop_first=True
)

X = combined.iloc[:len(train)]
X_test = combined.iloc[len(train):]

y = train["Survived"]

# =========================
# TRAIN / VALIDATION SPLIT
# =========================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# MODEL
# =========================

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# VALIDATION
# =========================

val_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, val_pred)

print("\nValidation Accuracy:", round(accuracy, 4))

# Cross Validation Score
cv_score = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Accuracy:", round(cv_score.mean(), 4))

# =========================
# TEST PREDICTIONS
# =========================

test_predictions = model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_predictions
})

submission.to_csv(
    "../submission.csv",
    index=False
)

print("\nsubmission.csv created successfully!")
print(submission.head())

Train Shape: (891, 12)
Test Shape: (418, 11)

Validation Accuracy: 0.8156
Cross Validation Accuracy: 0.8294

submission.csv created successfully!
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
